In [28]:
import json
from typing import *
from loader import load_training_problem, list_training_problems

data = list_training_problems()
problem_id = data[0]
example = load_training_problem(problem_id)

In [ ]:
def rotate_90(grid: List[List[int]]) -> List[List[int]]:
    rows, cols = len(grid), len(grid[0])
    rotated = [[0] * rows for _ in range(cols)]
    for i in range(rows):
        for j in range(cols):
            rotated[j][rows - 1 - i] = grid[i][j]
    return rotated

def rotate_180(grid: List[List[int]]) -> List[List[int]]:
    return [row[::-1] for row in grid[::-1]]

def rotate_270(grid: List[List[int]]) -> List[List[int]]:    
    rows, cols = len(grid), len(grid[0])
    rotated = [[0] * rows for _ in range(cols)]
    for i in range(rows):
        for j in range(cols):
            rotated[cols - 1 - j][i] = grid[i][j]
    return rotated

def flip_vertical(grid: List[List[int]]) -> List[List[int]]:
    return grid[::-1]

def flip_horizontal(grid: List[List[int]]) -> List[List[int]]:
    return [row[::-1] for row in grid]

def double_flip(grid: List[List[int]]) -> List[List[int]]:
    return flip_horizontal(flip_vertical(grid))

def apply_color_permutation(grid: List[List[int]], color_map: Dict[int, int] = None) -> List[List[int]]:
    import random
    if color_map is None:
        colors = list(range(10))
        shuffled_colors = colors.copy()
        while True:
            random.shuffle(shuffled_colors)
            if all(original != shuffled for original, shuffled in zip(colors, shuffled_colors)):
                break
        color_map = dict(zip(colors, shuffled_colors))
    return [[color_map.get(cell, cell) for cell in row] for row in grid]

groups = {
    "rotate": [None, rotate_90, rotate_180, rotate_270],
    "flip": [None, flip_vertical, flip_horizontal, double_flip],
    "color": [None, apply_color_permutation]
}

def assign_random_augmentations(seed: int = None) -> List[List[int]]:
    import random
    if seed is not None:
        random.seed(seed)
    chosen_augmentations = []
    for group_name, augmentations in groups.items():
        selected_augmentation = random.choice(augmentations)
        if selected_augmentation is not None:
            chosen_augmentations.append(selected_augmentation)
    return chosen_augmentations

def apply_augmentations_to_grids(grids: List[List[List[int]]], 
                                augmentations: List[Callable]) -> List[List[List[int]]]:
    augmented_grids = grids.copy()
    for augmentation in augmentations:
        if augmentation == apply_color_permutation:
            import random
            colors = list(range(10))
            shuffled_colors = colors.copy()
            while True:
                random.shuffle(shuffled_colors)
                if all(original != shuffled for original, shuffled in zip(colors, shuffled_colors)):
                    break
            color_map = dict(zip(colors, shuffled_colors))
            augmented_grids = [augmentation(grid, color_map) for grid in augmented_grids]
        else:
            augmented_grids = [augmentation(grid) for grid in augmented_grids]
    
    return augmented_grids


In [53]:
def create_placeholder(problem_data: Dict[str, Any], ground_truth:List[List[int]]) -> str:
    import random
    target_height = len(ground_truth)
    target_width = len(ground_truth[0]) if target_height > 0 else 0
    sampled_train = problem_data.get('train', [])
    all_matrices = []
    for example in sampled_train:
        all_matrices.append(example['input'])
        all_matrices.append(example['output'])
    test_examples = problem_data.get('test', [])
    for example in test_examples:
        all_matrices.append(example['input'])
    all_train_examples = []
    if 'train' in problem_data:
        all_train_examples.extend(problem_data['train'])
    if 'arc-gen' in problem_data:
        all_train_examples.extend(problem_data['arc-gen'])
    for example in all_train_examples:
        if example not in sampled_train:
            all_matrices.append(example['input'])
            all_matrices.append(example['output'])
    compatible_matrices = []
    for matrix in all_matrices:
        if len(matrix) == target_height and (not matrix or len(matrix[0]) == target_width):
            compatible_matrices.append(matrix)
    strategy = random.choices(['zeros', 'matrix', 'modified_matrix', 'ground_truth'], weights=[1/4, 1/4, 1/4, 1/4])[0]
    if strategy == 'zeros' or (strategy in ['matrix', 'modified_matrix'] and not compatible_matrices):
        placeholder_matrix = [[0] * target_width for _ in range(target_height)]
    elif strategy == 'matrix':
        chosen_matrix = random.choice(compatible_matrices)
        placeholder_matrix = [[cell for cell in row] for row in chosen_matrix]
    elif strategy == 'modified_matrix':
        chosen_matrix = random.choice(compatible_matrices)
        placeholder_matrix = [[cell for cell in row] for row in chosen_matrix]
        num_modifications = random.randint(1, 30)
        total_pixels = target_height * target_width
        num_modifications = min(num_modifications, total_pixels)
        positions = [(i, j) for i in range(target_height) for j in range(target_width)]
        positions_to_modify = random.sample(positions, num_modifications)
        for i, j in positions_to_modify:
            placeholder_matrix[i][j] = random.randint(0, 9)
    else:
        gt = ground_truth
        if gt is not None and len(gt) == target_height and (not gt or len(gt[0]) == target_width):
            placeholder_matrix = [[cell for cell in row] for row in gt]
        else:
            placeholder_matrix = [[0] * target_width for _ in range(target_height)]
    return placeholder_matrix

In [56]:
test_sample = example["train"][0]["input"]
test_sample_2 = example["train"][0]["output"]
generated_grid = create_placeholder(example, test_sample_2)
apply_augmentations_to_grids([test_sample, test_sample_2, generated_grid], assign_random_augmentations())

[[0, 7, 7], [7, 7, 7], [0, 7, 7]]
[[0, 0, 0, 0, 7, 7, 0, 7, 7], [0, 0, 0, 7, 7, 7, 7, 7, 7], [0, 0, 0, 0, 7, 7, 0, 7, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7], [0, 0, 0, 0, 7, 7, 0, 7, 7], [0, 0, 0, 7, 7, 7, 7, 7, 7], [0, 0, 0, 0, 7, 7, 0, 7, 7]]
[[0, 0, 0, 0, 6, 0, 3, 0, 0], [0, 0, 0, 6, 6, 8, 0, 0, 0], [0, 8, 0, 6, 3, 6, 0, 0, 0], [0, 6, 0, 0, 1, 0, 0, 6, 0], [6, 6, 6, 6, 6, 6, 8, 5, 9], [8, 6, 6, 6, 6, 6, 6, 6, 6], [2, 6, 0, 2, 6, 0, 0, 6, 0], [6, 5, 6, 6, 4, 6, 6, 6, 6], [6, 0, 6, 6, 6, 6, 6, 6, 4]]


[[[0, 7, 7], [7, 7, 7], [0, 7, 7]],
 [[0, 0, 0, 0, 7, 7, 0, 7, 7],
  [0, 0, 0, 7, 7, 7, 7, 7, 7],
  [0, 0, 0, 0, 7, 7, 0, 7, 7],
  [0, 7, 7, 0, 7, 7, 0, 7, 7],
  [7, 7, 7, 7, 7, 7, 7, 7, 7],
  [0, 7, 7, 0, 7, 7, 0, 7, 7],
  [0, 0, 0, 0, 7, 7, 0, 7, 7],
  [0, 0, 0, 7, 7, 7, 7, 7, 7],
  [0, 0, 0, 0, 7, 7, 0, 7, 7]],
 [[6, 0, 6, 6, 6, 6, 6, 6, 4],
  [6, 5, 6, 6, 4, 6, 6, 6, 6],
  [2, 6, 0, 2, 6, 0, 0, 6, 0],
  [8, 6, 6, 6, 6, 6, 6, 6, 6],
  [6, 6, 6, 6, 6, 6, 8, 5, 9],
  [0, 6, 0, 0, 1, 0, 0, 6, 0],
  [0, 8, 0, 6, 3, 6, 0, 0, 0],
  [0, 0, 0, 6, 6, 8, 0, 0, 0],
  [0, 0, 0, 0, 6, 0, 3, 0, 0]]]